# Writing in the Air — full experiment run on Colab

Runs every experiment of the project in order and downloads the results.

1. **Runtime → Change runtime type → T4 GPU**, then run the cells top to bottom.
2. Total time on a T4 is roughly 15–25 minutes (6 CNN trainings + 3 logistic regressions + fine-tuning).
3. Every number and figure is produced by the scripts in the repository and written to `results/` and `figures/`. Nothing is computed in this notebook itself.

In [ ]:
# Check that a GPU is available (training also works on CPU, just slower).
import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.is_available() and torch.cuda.get_device_name(0))

## 1. Get the code

**Option A** – clone the GitHub repository (fill in the URL; for a private repo use a personal access token).
**Option B** – upload `code.zip` (the project folder zipped) with the file browser on the left, then run the unzip line instead.

In [ ]:
REPO_URL = "https://github.com/<user>/airwrite.git"   # <- fill in

!rm -rf airwrite
!git clone -q {REPO_URL} airwrite
# Option B instead:  !unzip -q -o code.zip -d airwrite
%cd airwrite
!ls

## 2. Our own data

`data/raw/<writer>/<digit>/*.png` is committed to the repository, so after cloning it is already here.
If you uploaded the code as a zip without data, upload `data.zip` and unzip it into `data/raw/`.

If no data has been collected yet, set `SYNTHETIC = True`: the pipeline then runs on **stand-in data made from distorted MNIST digits**. Those numbers go to `results/synthetic/` and must **not** be reported as results on our dataset.

In [ ]:
import pathlib
n_images = len(list(pathlib.Path("data/raw").glob("*/*/*.png")))
print("collected images found:", n_images)

SYNTHETIC = n_images == 0
FLAG = "--synthetic" if SYNTHETIC else ""
print("running on", "SYNTHETIC stand-in data" if SYNTHETIC else "our collected data")

## 3. Train on MNIST
Logistic regression, CNN, and CNN with augmentation, each with seeds 42, 43, 44.

In [ ]:
!python train.py

## 4. Experiments on our data
Zero-shot ablation over preprocessing variants, fine-tuning on 70 %, leave-one-writer-out.

In [ ]:
!python ablation.py {FLAG}

## 5. Figures and headline numbers

In [ ]:
!python analysis.py {FLAG}

In [ ]:
from IPython.display import Image, display
fig_dir = pathlib.Path("figures/synthetic" if SYNTHETIC else "figures")
for name in ["learning_curves", "samples_by_writer", "ablation_cnn", "ablation_models",
             "confusion_ours_minimal", "confusion_ours_full", "lowo",
             "domain_stroke_width", "domain_fill_ratio", "misclassified_minimal"]:
    for folder in (fig_dir, pathlib.Path("figures")):
        path = folder / f"{name}.png"
        if path.exists():
            print(name); display(Image(str(path), width=650)); break

## 6. Download everything
`results/`, `figures/` and `models/` in one zip. Copy them back into the project folder on your laptop (and commit `results/` and `figures/`), so the report quotes exactly these files.

In [ ]:
!zip -q -r outputs.zip results figures models
from google.colab import files
files.download("outputs.zip")